# Concours MTH3302 A2024 
### Simon Bachand, Jérémie Bolduc, Jacqueline Koch, Julien Roux

## Prédiction de la consommation en carburant de voitures récentes.

### Contexte
Une gestion efficace de la consommation de carburant devient un enjeu crucial tant pour les conducteurs que pour l’industrie automobile, particulièrement dans le contexte actuel de transition énergétique et de réduction des émissions de gaz à effet de serre. La consommation en carburant des véhicules récents dépend de plusieurs caractéristiques techniques telles que la boîte de vitesses, la cylindrée, le nombre de cylindres et le type de transmission. Ces variables influencent directement l'efficacité énergétique et peuvent varier d’un véhicule à l’autre.

### Objectif

Dans cette étude, nous nous concentrons sur la prédiction de la consommation en carburant de voitures récentes. À partir d’un jeu de données comprenant la consommation moyenne en litres pour 100 kilomètres (L/100km) de près de 400 véhicules, ainsi que leurs caractéristiques techniques, l’objectif est de prédire la consommation en carburant pour un ensemble de test en fonction de ces différentes caractéristiques. Ce modèle prédictif permettra d’évaluer plus précisément les performances de consommation des véhicules et d'aider à identifier les facteurs déterminants pour l’optimisation de la consommation de carburant.

### Variables

La variable d'intérêt est la **consommation** en L/100km.

Les variables explicatives sont les suivantes:
- année: l'année du modèle
- type: le type de véhicule
- nombre_cylindres: le nombre de cylindres du moteur
- cylindrée: la cylindrée du moteur en L
- transmission: le type de transmission (propulsion, traction, 4x4 et intégrale)
- boite: le type de boite de vitesses (automatique ou manuelle)



## Chargement des données et des librairies

Importation des librairies utilisées dans le calepin.

In [ ]:
using CSV
using GLM
using MLJ
using DataFrames
using Gadfly
using StatsPlots
using Random
using Statistics
using Combinatorics
using LinearAlgebra
using HypothesisTests
using StatsModels
using CategoricalArrays
using StatsBase
using Turing

## Fonctions utilitaires

Les fonctions utilitaires ci-dessous sont utilisées dans le calepin pour effectuer certaines transformations sur les données (*One hot encoding*). 

In [ ]:
function one_hot_encode(data, columns)
    for column in columns
        coerce!(data, column => Multiclass)
    end

    one_hot = machine(OneHotEncoder(), data)
    fit!(one_hot, verbosity=0)
    return MLJ.transform(one_hot, data)
end

function encode(data, nominal_features, continuous_features, ordinal_features)
    for feature in nominal_features
        coerce!(data, feature => Multiclass)
    end

    mach = machine(OneHotEncoder(), data)
    fit!(mach, verbosity=0)
    data = MLJ.transform(mach, data)

    for feature in continuous_features
        coerce!(data, feature => MLJ.Continuous)
    end

    mach = machine(ContinuousEncoder(), data)
    fit!(mach, verbosity=0)
    data = MLJ.transform(mach, data)

    for feature in ordinal_features
        data[!, feature] = categorical(data[!, feature], ordered=true)
    end

    return data
end

function get_updated_features(data, features) 
    return vec(reduce(vcat, [
        filter(name -> startswith(name, string(feature)), names(data))
        for feature in features
    ]))
end

Les fonctions ci-dessous sont des méthodes classiques de de traitement des valeurs aberrantes, IRQ n'est plus utilisée dans ce calepin mais est laissée pour référence.

In [ ]:
function remove_outliers_STUD(data, features::Vector{Symbol}, target::Symbol=:y, threshold=3)
    formula = Term(target) ~ sum(Term(Symbol(feature)) for feature in features)

    model = lm(formula, data)

    y = data[:, target]
    X = modelmatrix(model)[:,2]
    H = X * inv(X' * X) * X'
    leverage = diag(H)

    predictions =  StatsModels.predict(model)
    residuals = y .- predictions

    student_residuals = residuals ./ sqrt.(1 .- leverage)
    outliers = abs.(student_residuals) .> threshold
    data = data[.!outliers, :]

    return data
end

# Supprimer les valeurs aberrantes avec la méthode de l'écart interquartile
function categorical_remove_outliers_IRQ(data, column, α=1.5)
    unique_values = unique(data[:, column])
    for value in unique_values
        subset = data[data[:, column] .== value, :]
        if size(subset, 1) == 0
            continue
        end
        q1 = quantile(subset[:, :consommation], 0.25)
        q3 = quantile(subset[:, :consommation], 0.75)
        iqr = q3 - q1
        lower_bound = q1 - α * iqr
        upper_bound = q3 + α * iqr
        data = data[(data[:, column] .!= value) .| ((data[:, column] .== value) .& (data[:, :consommation] .>= lower_bound) .& (data[:, :consommation] .<= upper_bound)), :]
    end
    return data
end

function categorical_replace_outliers_with_median(data, column, α=1.5)
    unique_values = unique(data[:, column])
    for value in unique_values
        subset = data[data[:, column] .== value, :]
        if size(subset, 1) == 0
            continue
        end
        q1 = quantile(subset[:, :consommation], 0.25)
        q3 = quantile(subset[:, :consommation], 0.75)
        iqr = q3 - q1
        lower_bound = q1 - α * iqr
        upper_bound = q3 + α * iqr

        outliers = (subset[:, :consommation] .< lower_bound) .| (subset[:, :consommation] .> upper_bound)
        non_outliers = subset[.!outliers, :]
        median_value = mean(non_outliers[:, :consommation])

        subset[outliers, :consommation] .= median_value
        data[data[:, column] .== value, :] = subset
    end
    return data
end

Les fonctions ci-dessous sont des méthodes classiques de de traitement des valeurs aberrantes, IRQ n'est plus utilisée dans ce calepin, mais est laissée pour référence.

Charger les données d'entrainement et de test.

In [ ]:
train = CSV.read("train.csv", DataFrame, decimal=',')
test = CSV.read("test.csv", DataFrame, decimal=',')

first(train, 5)

## Exploration des données

Nous avons commencé par examiner la structure du jeu de données d'entraînement pour mieux comprendre les variables disponibles et leurs types. À cette étape, l'objectif était de nous familiariser avec les données afin de déterminer leur pertinence et leur capacité à nous aider dans notre analyse. 

In [ ]:
describe(train)

Les données incluent des variables qui peuvent être utilisées pour prédire la consommation de carburant de différentes manières.

**Variables numériques (comme l'année de fabrication, le nombre de cylindres, la cylindrée) :**

Ces variables peuvent être utilisées pour comprendre l'impact de la technologie et de la taille du moteur sur la consommation. Par exemple, un moteur plus puissant (avec un nombre de cylindres plus élevé ou une cylindrée plus grande) pourrait être lié à une consommation plus élevée.

**Variables catégorielles (comme le type de véhicule, la transmission, et la boîte de vitesses) :**

Ces variables peuvent influencer la consommation en fonction du type de conduite. Par exemple, un véhicule avec une transmission 4x4 pourrait consommer plus de carburant que celui avec une transmission à traction. De même, la présence d'une boîte automatique peut avoir un impact sur la consommation en fonction des conditions de conduite.

Dans la partie 1, nous avons analysé l'influence de chaque variable sur la consommation.

---

Nous avons aussi utilisé un histogramme pour mieux comprendre la distribution des valeurs de consommation de carburant dans l'ensemble de données. Cette visualisation permet d'identifier la tendance centrale, l'étalement et la répartition de la consommation de carburant.

In [ ]:
Gadfly.plot(train, x=:consommation, Geom.histogram(bincount=15), Guide.xlabel("Consommation en L/100km"), Guide.ylabel("Consommation"))

La forme de l'histogramme suggère un pic autour du milieu de l'échelle, ce qui indique que la plupart des voitures ont une consommation de carburant modérée. La hauteur des barres diminue au fur et à mesure que l'on se rapproche des extrêmes inférieurs et supérieurs, ce qui indique qu'il y a moins de voitures ayant une consommation de carburant très faible ou très élevée. On peut reconnaître la forme d'une distribution normale. La distribution semble quelque peu symétrique, mais il y a une baisse notable de la fréquence au milieu, ce qui suggère la possibilité d'une irrégularité. Dans l'ensemble, la majorité des voitures se regroupent autour 10L/100km.

Afficher des diagrammes en boîte de consommation pour les variables catégoriques et un nuage de points pour la variable continue cylindrée

In [ ]:
function plot_variables(data)
    Gadfly.set_default_plot_size(30cm, 35cm)
    p1 = Gadfly.plot(data, x=:annee, y=:consommation, Geom.boxplot, Guide.title("Consommation par Année"), Guide.xlabel("Année"), Guide.ylabel("Consommation"))
    p2 = Gadfly.plot(data, x=:type, y=:consommation, Geom.boxplot, Guide.title("Consommation par Type de voiture"), Guide.xlabel("Type de voiture"), Guide.ylabel("Consommation"))
    p3 = Gadfly.plot(data, x=:nombre_cylindres, y=:consommation, Geom.boxplot, Guide.title("Consommation par nombre de cylindre"), Guide.xlabel("Nombre de cylindre"), Guide.ylabel("Consommation"))
    p4 = Gadfly.plot(data, x=:transmission, y=:consommation, Geom.boxplot, Guide.title("Consommation par type de transmission"), Guide.xlabel("Type de transmission"), Guide.ylabel("Consommation"))
    p5 = Gadfly.plot(data, x=:cylindree, y=:consommation, Geom.point, Geom.smooth(method=:lm), Guide.title("Nuage de points avec la droite de régression de consommation par cylindree"), Guide.xlabel("Cylindree"), Guide.ylabel("Consommation"))
    p6 = Gadfly.plot(data, x=:boite, y=:consommation, Geom.boxplot, Guide.title("Consommation par type de boite de vitesse"), Guide.xlabel("Boite de vitesse"), Guide.ylabel("Consommation"))

    grid = vstack(hstack(p1, p2), hstack(p3, p4), hstack(p5, p6))
    display(grid)

    # réinitialiser la taille pour ne pas affecter les autres graphiques
    Gadfly.set_default_plot_size(20cm, 15cm)
end

In [ ]:
plot_variables(train)

Ces différents graphiques illustrent la relation entre la consommation et chaque variable du jeu de données.   
On peut noter que la consommation suit une relation relativement linéaire avec la cylindrée et le nombre de cylindres ce qui peut indiquer qu'une régression linéaire pourrait être un bon modèle pour prédire la consommation.  

On note aussi la présence de **valeurs aberrantes** dans les données, représentées par des points isolés dans les graphiques.  
Ces valeurs aberrantes peuvent affecter la performance du modèle de régression linéaire. Il faudra les traiter avant de construire le modèle.   

---

# Partie 1
## Régressions linéaires simples

Afin d'analyser l'influence des différentes variables sur la consommation, nous avons effectué des régressions linéaires simples.

Pour chacune des six variables, nous avons vérifié les hypothèses 1 et 2 (linéarité et homoscédasticité des erreurs) à l'aide de graphiques.

L'hypothèse 3 (indépendance des erreurs) ne peut pas être vérifiée uniquement sur la base des données. Mais comme il s'agit ici d'un ensemble de données sur la consommation des voitures, où il ne devrait pas y avoir d'erreurs importantes, nous supposons que l'hypothèse est vérifiée.

Pour vérifier l'hypothèse 4 (distribution normale des erreurs), nous avons appliqué le test de Shapiro-Wilk, que nous avons appris dans un autre cours.

Avec la valeur $R^2$ nous avons testé si les variables ont un pouvoir explicatif significatif sur la consommation d'essence d'une voiture.

In [ ]:
function compute_residuals(model, y)
    ŷ = StatsModels.predict(model)
    res = (y - ŷ) / std(ŷ)

    return res
end

function plot_explanatory_variable(model, data, xlabel)
    predictions = StatsModels.predict(model)

    Gadfly.plot(
        x = data.x, 
        y = data.y,
        layer(
            x = data.x,
            y = predictions,
            Geom.line,
            Theme(default_color="red"),
        ),
        Geom.point,
        Guide.xlabel(xlabel), 
        Guide.ylabel("Consommation d'essence (L/100km)", orientation=:vertical),
    )
end

function shapiro_wilk_test(model, data)
    errors = compute_residuals(model, data.y)
    p = pvalue(ShapiroWilkTest(errors))

    if p > 0.05
        println("$p > 0.05 -> On accepte l'hypothèse que les données proviennent d'une distribution normale")
    else 
        println("$p ≤ 0.05 -> On rejette l'hypothèse que les données proviennent d'une distribution normale")
    end
end

function residuals_vs_predicted_values_plot(model, data)
    errors = compute_residuals(model, data.y)
    fitted_values = fitted(model)

    Gadfly.plot(
        layer(x=fitted_values, y=errors, Geom.point),
        Guide.xlabel("Valeurs prédites"),
        Guide.ylabel("Résidus"),
        Guide.title("homoscédasticité des erreurs"),
    )
end

function residuals_vs_index_plot(model, data)
    errors = compute_residuals(model, data.y)

    Gadfly.plot(
        layer(x=1:length(errors), y=errors, Geom.point),
        Guide.xlabel("Index"),
        Guide.ylabel("Résidus"),
        Guide.title("homoscédasticité des erreurs"),
    )
end

function linear_model(data, features::Vector{Symbol}, target::Symbol=:y)
    formula = Term(target) ~ sum(Term(feature) for feature in filter(x -> x != target, features))

    model = lm(formula, data)

    return model
end

# Cette fonction assure que les deux ensembles passé contiennent les mêmes valeurs catégorielles
function align_categorical_features(train, valid, features)
    data = vcat(train, valid)

    categorical_features = filter(x -> eltype(data[:, x]) <: AbstractString || eltype(data[:, x]) <: CategoricalValue, features)
    for feature in categorical_features
        train_values = unique(train[!, feature])
        valid_values = unique(valid[!, feature])

        train_indices_to_remove = findall(x -> !(x in valid_values), train[!, feature])
        train = train[setdiff(1:nrow(train), train_indices_to_remove), :]

        valid_indices_to_remove = findall(x -> !(x in train_values), valid[!, feature])
        valid = valid[setdiff(1:nrow(valid), valid_indices_to_remove), :]
    end

    return train, valid
end

In [ ]:
function plot_analyse(model, data, feature)
    Gadfly.set_default_plot_size(38cm, 10cm)

    p1 = plot_explanatory_variable(model, data, feature)
    p2 = residuals_vs_predicted_values_plot(model, data)
    p3 = residuals_vs_index_plot(model, data)

    grid = hstack(p1, p2, p3)
    display(grid)
end

In [ ]:
function linear_model(data, features::Vector{Symbol}, target::Symbol=:y)
    formula = Term(target) ~ sum(Term(feature) for feature in filter(x -> x != target, features))

    model = lm(formula, data)

    return model
end

## 1.1 Analyse de la variable _nombre_cylindres_

In [ ]:
x = float.(train.nombre_cylindres)
data = DataFrame(y = train.consommation, x = x)
model = linear_model(data, [:x])

**Vérification de l'hypothèse de linéarité et de l'hypothèse d'homoscédasticité des erreurs**

On observe une linéarité entre le nombre de cylindres du véhicule ainsi que sa consommation d'essence.

La droite coupe le nuage, alors l’hypothèse de linéarité est raisonnable.   
On peut voir que les erreurs ne sont pas constantes selon le nombre de cylindres. Cependant, cela pourrait être causé par la représentation de catégorie de nombre de cylindres disproportionnée.
À l'exception de quelques points aberrants, les résidus semblent bien distribués autour de 0.

In [ ]:
plot_analyse(model, data, "Nombre de cylindres")

**Vérification de l'hypothèse de normalité des erreurs**

Selon le test de Shapiro-Wilk, la P-valeur est inférieure à 0.05, les résidus ne sont pas distribués normalement 

In [ ]:
shapiro_wilk_test(model, data)

**Signifiance de la variable explicative**

Le nombre de cylindres a un pouvoir explicatif significatif sur la consommation d'essence d'une voiture.

In [ ]:
r2(model)

## 1.2 Analyse de la variable _type_

In [ ]:
data = DataFrame(y = train.consommation, x = train.type)
model = linear_model(data, [:x])

**Vérification de l'hypothèse de linéarité et de l'hypothèse d'homoscédasticité des erreurs**

Puisqu'il sagit d'une variable explicative catégorielle nominale, l'hypothèse de linéarité entre les catégories est automatiquement à rejeter.
On peut voir que les erreurs ne sont pas constantes selon le type de voiture, mais à l'exception de quelques points aberrants, les résidus semblent bien distribués autour de 0.

In [ ]:
plot_analyse(model, data, "Type")

**Vérification de l'hypothèse de normalité des erreurs**

Selon le test de Shapiro-Wilk, la P-valeur est inférieure à 0.05, les résidus ne sont donc pas distribués normalement.

In [ ]:
shapiro_wilk_test(model, data)

**Signifiance de la variable explicative**

Le type de la voiture a un pouvoir explicatif modéré sur la consommation d'essence d'une voiture.

In [ ]:
r2(model)

## 1.3 Analyse de la variable _cylindree_

In [ ]:
data = DataFrame(y = train.consommation, x = train.cylindree)
model = linear_model(data, [:x])

**Vérification de l'hypothèse de linéarité et de l'hypothèse d'homoscédasticité des erreurs**

On voit qu'il existe une relation linéaire entre la cylindrée du moteur et la consommation d'essence de la voiture.

La droite coupe le nuage, l'hypothèse de linéarité est donc raisonnable.
La dispersion des points autour de la droite semble relativement constante, à l'exception de quelques valeurs aberrantes et de la surreprésentation des moteurs de 2 litres de cylindrée. Cela indique que l'hypothèse 2 est vérifiée.

In [ ]:
plot_analyse(model, data, "Cylindrée")

**Vérification de l'hypothèse de normalité des erreurs**

Selon le test de Shapiro-Wilk, la P-valeur est inférieure à 0.05, les résidus ne sont donc pas distribués normalement.

In [ ]:
shapiro_wilk_test(model, data)

**Signifiance de la variable explicative**

La cylindrée du moteur a un pouvoir explicatif significatif sur la consommation d'essence d'une voiture.

In [ ]:
r2(model)

## 1.4 Analyse de la variable _transmission_

In [ ]:
x = train.transmission
data = DataFrame(y = train.consommation, x = x)
model = linear_model(data, [:x])

**Vérification de l'hypothèse de linéarité et de l'hypothèse d'homoscédasticité des erreurs**

Puisqu'il sagit d'une variable explicative catégorielle nominale, et non ordinale, l'hypothèse de linéarité entre les catégories est automatiquement à rejeter.

La variance de l'erreur semble être relativement constante pour chaque transmission observée et les résidus semblent bien répartis autour de 0.

In [ ]:
plot_analyse(model, data, "Transmission")

**Vérification de l'hypothèse de normalité des erreurs**

Selon le test de Shapiro-Wilk, la P-valeur est inférieure à 0.05, les résidus ne sont donc pas distribués normalement.

In [ ]:
shapiro_wilk_test(model, data)

**Signifiance de la variable explicative**

La transmission de la voiture a un pouvoir explicatif modéré sur la consommation d'essence d'une voiture.

In [ ]:
r2(model)

## 1.5 Analyse de la variable _boite_

In [ ]:
data = DataFrame(y = train.consommation, x = train.boite)
model = linear_model(data, [:x])

**Vérification de l'hypothèse de linéarité et de l'hypothèse d'homoscédasticité des erreurs**

Puisqu'il sagit d'une variable explicative catégorielle nominale, l'hypothèse de linéarité entre les catégories est automatiquement à rejeter.

La variance des erreurs semble être plus grande pour les véhicules automatiques que les véhicules manuels. Les résidus semblent bien distribués autour de 0.

In [ ]:
plot_analyse(model, data, "Boite")

**Vérification de l'hypothèse de normalité des erreurs**

Selon le test de Shapiro-Wilk, la P-valeur est inférieure à 0.05, les résidus ne sont donc pas distribués normalement.

In [ ]:
shapiro_wilk_test(model, data)

**Signifiance de la variable explicative**

La boîte de la voiture a un pouvoir explicatif très faible sur la consommation d'essence d'une voiture.

In [ ]:
r2(model)

## 1.6 Analyse de la variable _annee_

In [ ]:
x = train.annee
data = DataFrame(y = train.consommation, x = x)
model = linear_model(data, [:x])

**Vérification de l'hypothèse de linéarité et de l'hypothèse d'homoscédasticité des erreurs**

Il semble avoir une relation linéaire décroissante entre l'année et la consommation d'essence d'un véhicule.

La droite coupe le nuage, alors l’hypothèse de linéarité est raisonnable. 

La variance des erreurs varie d'année en année, mais les résidus semblent bien distribués autour de 0.

In [ ]:
plot_analyse(model, data, "Année")

**Vérification de l'hypothèse de normalité des erreurs**

Selon le test de Shapiro-Wilk, la P-valeur est inférieure à 0.05, les résidus ne sont donc pas distribués normalement.

In [ ]:
shapiro_wilk_test(model, data)

**Signifiance de la variable explicative**

L'année de la voiture a un pouvoir explicatif très faible sur la consommation d'essence.

In [ ]:
r2(model)

En résumé, on peut dire que le nombre de cylindres et la cylindrée ont un pouvoir explicatif significatif, le type et la transmission ont un pouvoir explicatif modéré. La boite et l'année n'ont qu'une faible influence.

Pour les variables numériques, l’hypothèse de linéarité est satisfaite.

Les erreurs sont indépendantes pour toutes les variables, ce qui est logique pour notre ensemble des données.

Aucune des variables satisfait l'hypothèse de la normalité des erreurs selon le test de Shapiro-Wilk.

On peut voir qu'aucune variable prise individuellement ne permet d'avoir une bonne régression linéaire.  
On va donc essayer de combiner plusieurs variables pour avoir un meilleur modèle.

# Partie 2
## Régressions linéaires multiples

Nous utiliserons cette fonction afin d'obtenir un jeu de données d'entraînement et de validation reproductible selon le seed.

In [ ]:
function get_data_set(seed, features, preprocess::Function = data -> nothing)
    Random.seed!(seed)
    data = CSV.read("train.csv", DataFrame, decimal=',')

    preprocess(data)

    train_id = sample(1:nrow(data), round(Int, .8*nrow(data)), ordered=true, replace=false)
    valid_id = setdiff(1:nrow(data), train_id)

    train = data[train_id,:]
    train = remove_outliers_STUD(train, features, :consommation)
    valid = data[valid_id,:]

    return train, valid
end

On met un seed pour avoir des résultats reproductibles et comparables.

In [ ]:
seed = 3456

### Régression linéaire multiple utilisant toutes les variables explicatives

In [ ]:
features = [:type, :transmission, :nombre_cylindres, :cylindree, :boite, :annee]
train, valid = get_data_set(seed, features)
ols_model = linear_model(train, features, :consommation)

ŷ = float.(StatsModels.predict(ols_model, valid))
y_valid = valid[:, :consommation]


println("R2 sur l’ensemble de validation est : ", r2(ols_model))
println("Le RMSE sur l’ensemble de validation est : ", rms(ŷ, y_valid))

### Régression linéaire multiple en utilisant que les variables explicatives ayant un pouvoir explicatif significatif

In [ ]:
features = [:type, :transmission, :cylindree, :nombre_cylindres]
train, valid = get_data_set(seed, features)
smaller_ols_model = linear_model(train, features, :consommation)

ŷ = float.(StatsModels.predict(smaller_ols_model, valid))
y_valid = valid[:, :consommation]

println("R2 sur l’ensemble de validation est : ", r2(smaller_ols_model))
println("Le RMSE sur l’ensemble de validation est : ", rms(ŷ, y_valid))

Test sur l'importance de la régression (calculer la valeur F) :  
On rejette l'hypothèse nulle, alors au moins une variable explicative possède un pouvoir prédictif significatif.  

In [ ]:
# F = (SSR/p) / (SSE/(n-p-1)) = (R2/p) / ((1-R2)/(n-p-1))
p = 6
n = size(valid, 1)
F = (r2(ols_model) / p) / ((1-r2(ols_model)) / (n-p-1))

c = quantile(FDist(p, n-p-1), 0.95)

if F > c
    println("On rejette l'hypothèse nulle.")
else
    println("On accepte l'hypothèse nulle.")
end

Comparaison des modèles : calculer les coefficients de détermination ajustés:  
Le $R^2_{aj}$ pour le modèle complet est 0.844 et le $R^2_{aj}$ pour le modèle réduit est 0.838.
Donc, si l'on tient compte du nombre de variables utilisées, un modèle réduit s'ajuste mieux à nos données.

In [ ]:
# Calculer R2 ajusté pour le modèle complet
p = 6
R2_adj = 1 - (1 - r2(ols_model)) * (n - 1) / (n - p - 1)

# Calculer R2 ajusté pour le modèle réduit
p = 4
R2_adj_sign = 1 - (1 - r2(smaller_ols_model)) * (n - 1) / (n - p - 1)

println("R2 ajusté pour le modèle complet: $R2_adj")
println("R2 ajusté pour le modèle réduit: $R2_adj_sign")


### Calcul du VIF pour le modèle de régression linéaire multiple


In [ ]:
features = [:type, :transmission, :boite, :nombre_cylindres, :cylindree, :annee]
nominal_features = [:type, :transmission, :boite]
continuous_features = [:nombre_cylindres, :cylindree, :annee]
ordinal_features = []

train, valid = get_data_set(seed, features)
data = vcat(train, valid)
data = encode(data, nominal_features, continuous_features, ordinal_features)
features = map(x -> Symbol(x), get_updated_features(data, features))

p = length(features)
vif = Dict()
for i in 1:p
    model = linear_model(data, continuous_features, features[i])
    vif[features[i]] = 1 / (1 - r2(model))
end

filter((kv) -> kv[2] > 8, vif)

On obtient un VIF autour de 8-9 pour la cylindrée et le nombre de cylindres, ce qui peut indiquer la présence d'une multicolinéarité problématique. Cela pourrait entraîner un surajustement aux données d'entraînement et augmenter la variance des estimateurs de coefficients de régression.  
La prédiction est donc plus instable et difficile.  

Puisque nous n'avons que très peu de variables explicatives, nous souhaiterions éviter de passer par une analyse des composantes principales, car cela réduirait encore plus notre jeu de données. Nous essaierons plutôt de contrôler cette potentielle multicolinéarité grâce avec la régression ridge.

Nous nous attaquerons à ce problème dans la **Partie 3**.

In [ ]:
# analyser la relation entre nombre des cylindres et cylindrée
set_default_plot_size(30cm, 10cm)
p1 = Gadfly.plot(train, x=:nombre_cylindres, y=:cylindree, Geom.point, Guide.title("Nuage de points entre nombre de cylindres et cylindrée"), Guide.xlabel("Nombre de cylindres"), Guide.ylabel("Cylindrée"))
p2 = Gadfly.plot(train, x=:cylindree, y=:nombre_cylindres, Geom.point, Guide.title("Nuage de points entre cylindrée et nombre de cylindres"), Guide.xlabel("Cylindrée"), Guide.ylabel("Nombre de cylindres"))

display(hstack(p1, p2))
# reset plotsize
Gadfly.set_default_plot_size(15cm, 10cm)

Dans les graphiques suivants, on peut voir qu’il y a une relation linéaire entre le nombre des cylindres et la cylindrée, ce qui confirme notre intuition donnée par les VIF.

Dans cette partie nous avons fait une régression linéaire multiple avec toutes les variables explicatives et une autre avec les variables explicatives qui ont le pouvoir explicatif le plus significatif.   
Cependant pour notre modèle final, nous chercherons à obtenir le meilleur RMSE possible, ainsi dans notre cas c'est le modèle complet qui nous a donné les meilleurs résultats.


# Partie 3
## Régression linéaire multiple avec technique de régularisation

In [ ]:
RidgeRegressor = @load RidgeRegressor pkg=MLJLinearModels verbosity=0 
function ridge_regression_cv(X, y, k_folds=5)
    lambdas = 10 .^ LinRange(-4, 4, 100)
    cv = CV(nfolds=k_folds)

    best_score = Inf
    λ̂ = nothing

    for lambda in lambdas
        model = RidgeRegressor(lambda=lambda)
        mach = machine(model, X, y)

        cv_result = evaluate!(mach, resampling=cv, verbosity=0)

        score = mean(cv_result.measurement)

        if score < best_score
            best_score = score
            λ̂ = lambda
        end
    end

    mach = machine(RidgeRegressor(lambda=λ̂) , X, y)
    fit!(mach, verbosity=0)

    return mach, λ̂
end

### Régression ridge avec validation croisée pour choix du lambda optimal

In [ ]:
features = [:type, :transmission, :boite, :nombre_cylindres, :cylindree, :annee]
nominal_features = [:type, :transmission, :boite]
continuous_features = [:nombre_cylindres, :cylindree, :annee]
ordinal_features = []

train, valid = get_data_set(seed, features)
train, valid = align_categorical_features(train, valid, features)

y_train = train[!, :consommation]
X_train = train[!, features]
X_train = encode(X_train, nominal_features, continuous_features, ordinal_features)

X_valid = valid[!, features]
X_valid = encode(X_valid, nominal_features, continuous_features, ordinal_features)

ridge_machine, λ̂ = ridge_regression_cv(X_train, y_train)

ŷ = MLJ.predict(ridge_machine, X_valid)
y_valid = valid[:, :consommation]

println("RMSE: $(rms(ŷ, y_valid))\nλ optimal: $λ̂")

On obtient, en apparence, un résultat de RMSE très comparable à celui de la régression linéaire multiple avec tous les paramètres.  
On peut aussi observer que le coefficient le plus important du modèle de régression linéaire multiple, à savoir, la cylindrée, a été pénalisé dans la régression ridge. Cependant, le lambda optimal obtenu est très petit, donc la régression ridge se comporte davantage comme une régression linéaire.

Tout cela suggère que la multicolinéarité dans les données n'est pas suffisamment élevée pour justifier l'utilisation d'une régularisation importante.

Après avoir écarté l'hypothèse de multicolinéarité, nous nous tournerons vers une approche de régression bayésienne pour tenter d'améliorer les prédictions du modèle de régression linéaire multiple dans la **Partie 4**.

In [ ]:
fitted_params(ridge_machine), ols_model

# Partie 4
## Régression bayésienne

Puisque nous encoderons nos variables explicatives nominales avec l'encodage one hot, il sera difficile de trouver une loi à priori informative pour celle-ci. 

Cependant, comme l'a montré notre analyse préliminaire, la cylindrée et le nombre de cylindres sont deux variables faciles à poser sur une échelle continue et qui ont un bon pouvoir explicatif sur la consommation d'essence.   
Nous nous concentrerons donc à trouver des lois a priori pour ces deux variables.

In [ ]:
# fonction pour centrer et réduire un vecteur
function standardize_vec(x)
    centered = x .- mean(x)
    return centered ./ std(x)
end

@model function bayesian_regression(X, y, priors)
    features = names(X)
    predictors = size(X, 2)

    α ~ Normal(mean(y), std(y))
    β = zeros(predictors)
    for i in 1:predictors
        if haskey(priors, features[i])
            β[i] ~ priors[features[i]]
        else 
            # nous supposerons une loi normal centrée en 0 avec une 
            # grande variance pour les variables pour lesquelles il
            # est difficile d'obtenir une loi a priori informative 
            β[i] ~ Normal(0, 3)
        end
    end
    if (haskey(priors, "σ²"))
        σ² ~ priors["σ²"]
    else 
        σ² ~ InverseGamma(1, 2)
    end

    μ = α .+ Matrix(X) * β
    Σ = σ² * I

    return y ~ MvNormal(μ, Σ)
end

### Traitement de données antérieures

In [ ]:
# source https://www.kaggle.com/datasets/eimadevyni/car-model-variants-and-images-dataset
data = CSV.read("cars_dataset.csv", DataFrame, decimal=',')

# On prend les données de 2010 à 2024 pour conserver un échantillon suffisamment grand
data = filter(x -> !ismissing(x.cylinders) && !ismissing(x.engine_specs_title) && x.from_year >= 2010 && x.to_year <= 2024, data)
data = filter(x -> begin
    m = match(r"\d(\.\d)?(?=L)", x.engine_specs_title)

    return m != nothing
end, data)
data = filter(x -> !ismissing(x.combined) && x.combined != nothing, data)

# extraction de la cylindrée
data.cylindree = map(x -> begin 
    m = match(r"\d(\.\d)?(?=L)", x)

    if m != nothing
        return parse(Float64, m.match)
    end
end, data.engine_specs_title)

# extraction du nombre de cylindres
data.nombre_cylindres = map(x -> begin 
    m = match(r"\d\d?", x)

    if m != nothing
        return parse(Float64, m.match)
    end
end, data.cylinders)

# extraction de la consommation d'essence en L/100km
data.consommation = map(x -> begin
    m = match(r"\d(\.\d)?\s*?(?=L\/100\s*Km)", x)

    if m != nothing
        return parse(Float64, m.match)
    end
end, data.combined)

data.annee = round.((data.from_year .+ data.to_year) ./ 2; digits=0)
data.annee = standardize_vec(data.annee);

### Loi a priori pour la cylindrée

Nous utiliserons la loi Gamma pour modéliser la loi a priori, car elle s'ajuste bien aux données de cylindrées qui sont assez asymétriques.

In [ ]:
Gadfly.set_default_plot_size(15cm, 10cm)

dist_cylindree = fit(Gamma, data.cylindree)

Gadfly.plot(
    layer(x->pdf(dist_cylindree, x), 0, 12, Theme(default_color=colorant"red")),
    layer(x=data.cylindree, Geom.histogram(bincount=30, density=true)),
)

### Loi a priori pour le nombre de cylindres

Nous utiliserons la loi Gamma pour modéliser la loi a priori, car elle s'ajuste bien aux données de nombre de cylindres qui sont aussi assez asymétriques.

In [ ]:
dist_nombre_cylindres = fit(Gamma, data.nombre_cylindres)

Gadfly.plot(
    layer(x->pdf(dist_nombre_cylindres, x), 0, 18, Theme(default_color=colorant"red")),
    layer(x=data.nombre_cylindres, Geom.histogram(bincount=30, density=true)),
)

### Choix de la loi a priori pour l'année

Nous avons décidé de centré réduire l'année afin de limiter l'impact de grand nombre sur le modèle et mieux exposer la linéarité entre l'année et la consommation d'essence.   
Nous modéliserons la distribution de celle-ci à l'aide d'une loi Normale.

In [ ]:
dist_annee = fit(Normal, data.annee)

Gadfly.plot(
    layer(x->pdf(dist_annee, x), -4, 4, Theme(default_color=colorant"red")),
    layer(x=data.annee, Geom.histogram(bincount=30, density=true)),
)

### Importance de la variance de la consommation pour la loi a priori de la variance

Puisque la variance est relativement grande (~7), nous modéliserons celle-ci à l'aide d'une loi InverseGamma de paramètres alpha = 1 et bêta = 2 pour avoir des ailes assez lourdes pour supporter de plus grandes variances

In [ ]:
var(data.consommation)

### Modèle de régression bayésienne

In [ ]:
features = [:type, :transmission, :boite, :nombre_cylindres, :cylindree, :annee]
nominal_features = [:type, :transmission, :boite]
continuous_features = [:nombre_cylindres, :cylindree, :annee]
ordinal_features = []

train, valid = get_data_set(seed, features)
train, valid = align_categorical_features(train, valid, features)

train.annee = standardize_vec(train.annee)
valid.annee = standardize_vec(valid.annee)

y_train = train[!, :consommation]
X_train = train[!, features]
# encodage one hot pour les variables nominales et continue pour les variables continues
X_train = encode(X_train, nominal_features, continuous_features, ordinal_features)

y_valid = valid[!, :consommation]
X_valid = valid[!, features]
X_valid = encode(X_valid, nominal_features, continuous_features, ordinal_features)

model = bayesian_regression(
    X_train, 
    y_train, 
    Dict(
        "cylindree" => dist_cylindree,
        "nombre_cylindres" => dist_nombre_cylindres,
        "annee" => dist_annee,
        "σ²" => InverseGamma(1, 2),
    )
)
# nous utiliserons l'algorithme d’échantillonnage de Gibbs couplé à celui No U-Turn Sampling avec un taux d'acceptation de 65% pour 
# les lois a posteriori trop complexes pour Gibbs.
# Bien que nous avons vu Metropolis-Hastings, cet méthode d’échantillonnage nous a proposé de meilleurs résultats
chain_sampler = Gibbs(NUTS(0.65))
chain = sample(model, chain_sampler, 1000, discard_initial = 100)

StatsPlots.plot(chain)

In [ ]:
# https://turinglang.org/docs/tutorials/05-linear-regression/
function bayesian_prediction(chain, X)
    params = get_params(chain)

    α̂ = params.α
    β̂ = reduce(hcat, params.β)
    targets = α̂' .+ Matrix(X) * β̂'

    # nous prendrons la moyenne de la loi a posteriori pour faire nos predictions
    return vec(mean(targets; dims=2))
end

In [ ]:
ŷ = bayesian_prediction(chain, X_valid)
rms(ŷ, y_valid)

### Analyse des résultats

On obtient une mesure de RMSE légèrement meilleure par rapport au modèle de régression linéaire multiple. Cela pourrait indiquer que nos lois a priori sont bien choisies et permettent de mieux capturer la structure des données.

Toutefois, l'amélioration n'est pas très significative. Cela peut-être du aux limitations qu'impose l'encodage one-hot, ce type de représentation traite chaque catégorie comme indépendante, ignorant les relations potentielles entre elles et rend la modélisation de lois a priori informatives pour celles-ci beaucoup plus difficile. Une solution possible à ce problème serait d'encoder dans un ordre particulier si possible (par exemple, tailles croissantes des types de véhicules) afin de les rendre ordinales, rendant le choix d'une loi a priori plus simple.

# Partie 5
Bien que les méthodes précédentes aient permis d'obtenir des résultats intéressants, il y a encore une marge d'amélioration.  
Dans cette partie, nous allons reprendre l'exploration des données pour identifier des relations ou des transformations qui pourraient améliorer la performance des modèles.

## Réduction de l'ensemble de données

In [ ]:
train = CSV.read("train.csv", DataFrame, decimal=',');

In [ ]:
println("Nombre de ligne du jeu de données d'entraînement : ", nrow(train))
println("Nombre de ligne unique du jeu de données : ", nrow(unique(train)))
println("Nombre de ligne unique du jeu de données sans l'année: ", nrow(unique(select(train, Not(:annee)))))
println("Nombre de valeur de consommation unique : ", nrow(unique(train, :consommation)))

Ces informations nous permettent de voir que le jeu de données contient beaucoup de lignes identiques, encore plus si on retire l'année.  
Cela indique que le modèle pourrait être biaisé par un poids plus important des données redondantes dans le modèle et que l'année semble ne pas apporter pas d'information significative.
  
De plus, on voit que les valeurs possibles de consommation sont limitées, ce qui peut indiquer que certaines combinaisons de caractéristique sont équivalentes.  

In [ ]:
unique_data = sort(unique(train), :consommation)
unique_data = sort((train), :consommation)

unique_data[!, :model] .= string.(unique_data[!, :type], "_", unique_data[!, :nombre_cylindres], "_", unique_data[!, :cylindree], "_", unique_data[!, :transmission], "_", unique_data[!, :boite])

Gadfly.set_default_plot_size(38cm, 25cm)

plt = Gadfly.plot(
    layer(unique_data, x=:model, y=:consommation, Geom.point),
    layer(unique_data, x=:model, y=GLM.predict(lm(@formula(consommation ~ model), unique_data)), Geom.line, Theme(default_color="red")),
    Guide.xlabel("Cylindrée"), Guide.ylabel("Consommation"),
)

display(plt)

À partir des différentes caractéristiques (hors année) nous avons créé une nouvelle colonne **Model**, qui est une chaîne de caractère qui représente les caractéristiques de la voiture.  
Puis nous avons utilisé cette nouvelle colonne pour afficher la distribution de la consommation avec la droite de régression.  
On peut voir que la consommation semble suivre une relation continue, celle-ci semble augmenter linéairement après les 3 premiers **Model** puis s’accélère sur la fin.  

Les derniers **Model** correspondent à des grosses voitures type SUV ou des voitures de sport (grosse cylindrée et grand nombre de cylindres) qui consomment beaucoup plus que des voitures standard.
C'est donc cohérent que la consommation augmente plus rapidement pour ces **Model**.  

De plus, on peut voir que pour une même **Model** on a parfois des consommations très différentes, étant donné que nous n'avons pas pris l'année pour créer le **Model** nous allons voir si l'année peut expliquer ces différences.


In [ ]:
sort(unique_data[unique_data[:,:model] .== "VUS_petit_4_2.0_integrale_automatique", :], :annee)

Ici nous affichons les lignes triées par année pour le **model** : *VUS_petit_4_2.0_integrale_automatique*  
On voit pas de relation évidente entre l'année et la consommation, on peut donc supposer que l'année n'apporte pas d'information significative pour prédire la consommation.

In [ ]:
sort(unique_data[unique_data[:,:model] .== "VUS_petit_4_2.4_traction_automatique", :], :annee)

Ici nous affichons les lignes triées par année pour le **model** : *VUS_petit_4_2.4_traction_automatique*  
Ici peut légèrement voir une tendance à la baisse de la consommation en fonction de l'année, mais pas de manière significative.

Cela montre que mous pourrions combiner les différentes consommations pour un même modèle pour réduire la base de données, car l'année n'apporte pas d'information très significative.  

In [ ]:
mean_consommation = combine(groupby(unique_data, :model), :consommation => mean => :consommation_mean)
unique_data_mean = innerjoin(unique(select(unique_data, Not(:annee))), mean_consommation, on=:model)

mode_consommation = combine(groupby(unique_data, :model), :consommation => mode)
unique_data_mode = innerjoin(unique(select(unique_data, Not(:annee))), mode_consommation, on=:model);

On va alors réduire la base de données en prenant la moyenne ou le mode de la consommation pour un même **model**.  

In [ ]:
Gadfly.set_default_plot_size(28cm, 14cm)

plt = Gadfly.plot(
    layer(unique_data_mean, x=:model, y=:consommation_mean, Geom.point),
    layer(unique_data_mean, x=:model, y=GLM.predict(lm(@formula(consommation_mean ~ model), unique_data_mean)), Geom.line, Theme(default_color="red")),
    Guide.xlabel("Cylindrée"), Guide.ylabel("Consommation"),
    Guide.xticks(label=false),
    Guide.title("Régression linéaire de la consommation moyenne par modèle")
)

display(plt)

plt = Gadfly.plot(
    layer(unique_data_mode, x=:model, y=:consommation_mode, Geom.point),
    layer(unique_data_mode, x=:model, y=GLM.predict(lm(@formula(consommation_mode ~ model), unique_data_mode)), Geom.line, Theme(default_color="red")),
    Guide.xlabel("Cylindrée"), Guide.ylabel("Consommation"),
    Guide.xticks(label=false),
    Guide.title("Régression linéaire de la consommation modale par modèle")
)

display(plt)

Ces graphiques nous montrent que prendre le *mode* laisse les certains pics de consommation alors que la moyenne lisse ces pics.  
Nous allons donc utiliser la consommation moyenne pour éviter le risque d'avoir un modèle détérioré par ces pics.  

In [ ]:
unique_data_mean[unique_data_mean[:,:model] .== "VUS_petit_4_2.0_integrale_automatique", :consommation_mean] .= 9.04654;
unique_data_mean[unique_data_mean[:,:model] .== "VUS_petit_4_2.0_traction_automatique", :consommation_mean] .= 8.40036
unique_data_mean[unique_data_mean[:,:model] .== "voiture_sous_compacte_6_3.0_integrale_automatique", :consommation_mean] .= 10.6914
unique_data_mean[unique_data_mean[:,:model] .== "voiture_sous_compacte_6_3.0_propulsion_automatique", :consommation_mean] .= 10.2265
unique_data_mean[unique_data_mean[:,:model] .== "VUS_petit_4_1.6_integrale_automatique", :consommation_mean] .= 7.3503125

On traite certaines valeurs aberrantes manuellement en les remplaçant par des valeurs plus cohérentes pour le **Model**  
On aurait pu le faire de manière automatique, mais comme cela ne concerne que quelques lignes nous avons préféré le faire manuellement.

In [ ]:
unique_data_mean = select(unique_data_mean, Not(:consommation, :model))
unique_data_mode = select(unique_data_mode, Not(:consommation, :model));

println("Taille du jeu de données d'entraînement après réduction: ", nrow(unique_data_mean))

In [ ]:
X₀_mean = coerce(select(unique_data_mean, Not(:consommation_mean)), :type=>Multiclass, :transmission=>Multiclass, :boite=>Multiclass)
X₀_mean[!, :nombre_cylindres] = Float64.(X₀_mean[!, :nombre_cylindres])

X₀_mode = coerce(select(unique_data_mode, Not(:consommation_mode)), :type=>Multiclass, :transmission=>Multiclass, :boite=>Multiclass)
X₀_mode[!, :nombre_cylindres] = Float64.(X₀_mode[!, :nombre_cylindres])

y₀_mean = unique_data_mean[!, :consommation_mean]
y₀_mode = unique_data_mode[!, :consommation_mode];

In [ ]:
LinearRegressor = @load LinearRegressor pkg=MLJLinearModels verbosity=0
function create_linear_machine(X, y)
    linear_machine = machine(LinearRegressor(), X, y)
    fit!(linear_machine , verbosity=0)
    return linear_machine
end

X₀_mean = one_hot_encode(X₀_mean, [:type, :transmission, :boite])
X₀_mode = one_hot_encode(X₀_mode, [:type, :transmission, :boite])

linear_machine_mean = create_linear_machine(X₀_mean, y₀_mean)
linear_machine_mode = create_linear_machine(X₀_mode, y₀_mode)

In [ ]:
evaluate!(linear_machine_mean, resampling=CV(shuffle=true), measure=rms)

In [ ]:
evaluate!(linear_machine_mode, resampling=CV(shuffle=true), measure=rms)

Les mesures de RMSE de notre modèle sur l'ensemble de validation pour la consommation sont comme prévu meilleur avec la consommation moyenne (utilisation de la Cross validation du modèle linéaire avec MLJ).

On peut néanmoins voir que le RMSE est plus élevé que pour d'autres modèles vus précédemment, mais en pratique sur l’ensemble de test, le modèle est plus performant, cela est du aux transformations sur notre ensemble qui ont permis de réduire l'impacte du bruit (ou valeur aberrante) et de mieux généraliser.  

Nous allons essayer les mêmes transformations avec le modèle bayésien pour voir si cela améliore les performances.

# Partie 6
## Régression bayésienne avec réduction de l'ensemble de données

In [ ]:
features = [:type, :transmission, :boite, :nombre_cylindres, :cylindree] 
categorical_features = [:type, :transmission, :boite]
continuous_features = [:nombre_cylindres, :cylindree]
ordinal_features = []

test = CSV.read("test.csv", DataFrame, decimal=',');

X_train = unique_data_mean[!, features]
y_train = unique_data_mean[!, :consommation_mean]
X_test = test[!, features]

X_train, X_test = align_categorical_features(X_train, X_test, features)

X_train = encode(X_train, categorical_features, continuous_features, ordinal_features)
X_test = encode(X_test, categorical_features, continuous_features, ordinal_features)

model = bayesian_regression(
    X_train, 
    y_train, 
    Dict(
        "cylindree" => dist_cylindree,
        "nombre_cylindres" => dist_nombre_cylindres,
        "σ²" => InverseGamma(1, 2),
    )
)
chain_sampler = Gibbs(NUTS())
chain = sample(model, chain_sampler, 2000, discard_initial = 1000)

In [ ]:
ŷ = bayesian_prediction(chain, X_test)

df_pred = DataFrame(id=1:size(test, 1), consommation=ŷ)
CSV.write("benchmark.csv", df_pred)

Nous utilisons le même modèle Bayésien que dans la partie précédente pour prédire la consommation.  
Cette méthode améliore les performances par rapport à la régression linéaire multiple de la partie précédente et est la meilleure méthode que nous avons trouvée pour prédire la consommation.

# Conclusion  

| Méthode  | Valeur de RMSE sur l'ensemble de test (Kaggle)  |
|---|---|
| 1. Régression linéaire simple  | Non testé (pas assez de potentiel) |
| 2. Régression linéaire multiple  | 0.93221 |
| 3. Régression ridge  | 0.95629 |
| 4. Régression bayésienne  | 0.92256  |
| 5. Réduction des données et linéaire multiple  | 0.88987  |
| 6. Réduction des données et Régression bayésienne  | 0.84811  |

Dans ce rapport nous avons évalué la performance de plusieurs méthodes dans le but de prédire le plus précisément possible les valeurs de consommation de différents véhicules à partir de leurs caractéristiques.  
En premier, nous avons essayé de faire une **régression linéaire simple** sur les différentes caractéristiques. Cette méthode a été rapidement écartée pour son manque de potentiel, car les variables indépendamment n'ont pas un pouvoir explicatif suffisant.  
La **régression linéaire multiple** a montré des résultats plus prometteurs, mais la possibilité de multicolinéarité entre les variables ou la présence de valeurs aberrantes ont limité la performance du modèle.  
Pour contrer le problème de multicolinéarité, nous avons utilisé la **régression ridge**, mais les résultats n'ont pas été meilleurs, ce qui nous a indiqué que la multicolinéarité n'était pas le problème principal.  
Nous avons ensuite utilisé la **régression bayésienne** pour essayer de capturer plus de la structure des données, les résultats ont été prometteurs, mais il reste encore de la place pour l'amélioration.  
Pour ces méthodes, nous utilisions principalement des méthodes généralistes telles que **IRQ** ou celle avec les ***Studentized residuals*** pour traiter les valeurs aberrantes, ces méthodes permettent de détecter une partie de ces valeurs, mais certaines sont plus subtiles et demandent une analyse plus poussée spécifique à nos données. 
C'est pourquoi nous avons repris l'exploration pour essayer de trouver des transformations ou des relations entre les variables qui pourraient améliorer la performance du modèle. Nous avons alors remarqué que certaines transformations pouvaient être faites pour réduire la taille de l’ensemble de données et réduire l'impacte de valeurs aberrantes.  

L’ajout de ce prétraitement apporte des gains significatifs à nos modèles, combinés à la régression linéaire multiple, le RMSE descend à 0.88987 et la meilleure performance est atteinte avec régression bayésienne, avec une RMSE de 0.84811.   

Ce projet nous a permis de voir l’importance de l’exploration des données et de la recherche de relations entre les variables pour améliorer nos prédictions et aussi de mieux comprendre le fonctionnement des modèles bayésiens dans un contexte réel.